Приветствую! Данный алгоритм предоставляет возможность прогнозирования исхода судебных дел с помощью обработки естественного языка (NLP) и модели машинного обучения - Random Forest. Также, алгоритм был дополнен использованием регулярных выражений и еще одной модели машинного обучения - K-Means, метод k-средних, позволяющей добиться более точного результата.

Ознакомиться с концептуальной моделью и работой алгоритма в виде схем, можно в прикрепленном pdf файле под названием schemes.

Датасет был составлен из 100 судебных дел арбитражных судов РФ с помощью информационной системы «Мой Арбитр» (https://my.arbitr.ru/). 

Рассматривались только завершенные решения в полном объеме из Банка Решений. Большинство решений относятся к периоду марта 2025 года, но есть и решения 2017, 2018, 2019 годов. Также рассматриваются мотивированные решения. Смотреть вероятность выигрыша/проигрыша той или иной стороны или удовлетворении иска/заявления можно будет на незавершенных решениях.

Ввиду проблем с фронтэндом сайта «Мой Арбитр», пришлось отказаться от автоматического сбора данных, файлы судебных дел были скачаны вручную.

Ознакомиться с библиотеками, требуемыми для работы программы и их версиями, можно с помощью файла requirements.txt.

Ниже предоставлен модуль important_features. Модуль с помощью регулярных выражений и NLP первоначально очищает текст судебного дела и находит судью.

Стоит отметить, что русскоязычные тексты гораздо сложнее в плане выделения основных признаков, поэтому частично разбиралась структура документа с помощью регулярных выражений.

In [3]:
import logging
import time
import re
import pymorphy3
from natasha import (
    Segmenter,
    MorphVocab,
    
    NewsEmbedding,
    NewsMorphTagger,
    NewsSyntaxParser,
    NewsNERTagger,
    
    PER,
    NamesExtractor,

    Doc
)
import os
import fitz


os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

segmenter = Segmenter()
morph_vocab = MorphVocab()

emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)
syntax_parser = NewsSyntaxParser(emb)
ner_tagger = NewsNERTagger(emb)

names_extractor = NamesExtractor(morph_vocab)

# Инициализация морфологического анализатора
morph = pymorphy3.MorphAnalyzer()


def remove_strings_with_english(text_list):
    # Регулярное выражение для поиска английских букв
    return [text for text in text_list if not re.search(r'[a-zA-Z]', text)]

def remove_non_russian_alpha_lines(lines):
    return [line for line in lines if re.match("^[А-яёЁ]+$", line)]

def remove_lines_with_multiple_spaces(lines):
    return [line for line in lines if not re.search(r"\s{2,}", line)]


# Пример списка стоп-слов
stop_words_set = {"я", "ты", "он", "она", "оно", "мы", "вы", "они", "не", "нет", "того" "это", "тот", "та", "те", "который", "чей", "кто", "что", 
                      "да", "а", "но", "и", "или", "да", "также", "же", "ли", "бы", "для", "от", "из", "с", "на", 
                      "в", "по", "к", "у", "о", "об", "при", "для", "за", "перед", "после", "до", "через", "между", "над", 
                      "под", "вокруг", "из-за", "около", "через", "вон", "про", "между", "если", "когда", "пока", "хотя", 
                      "как", "так", "потому что", "чтобы", "ибо", "то есть", "где", "когда", "куда", "откуда", "уж", "вот", "вот такой", 
                      "именно", "вот это", "тот", "всё", "вся", "все", "некоторый", "несколько", "один", "два", "три", "первый", "второй", 
                      "последний", "каждый", "любой", "любой", "даже", "довольно", "вовсе", "только", "никогда", "всегда", "потом", "когда-то", 
                      "везде", "впрочем", "мало", "много", "больше", "меньше", "слишком", "скорее", "уж", "как бы", "например", "например", 
                      "возможно", "следует", "конечно", "вроде", "чем", "что-то", "тот", "этот", "такой", "никакой", "другой", "так как", 
                      "а вот", "пусть", "либо", "просто", "типо", "короче", "хотя бы", "и так далее", "далее", "есть", "потому", "то", "поскольку", "б"}


def CleanText(directory: str, filename: str):
    start_time = time.time()
    logging.info("Загрузка файла...")
    doc = fitz.open(directory+filename)
    text = "\n".join([page.get_text() for page in doc])

    text = text.replace('\n', '')
    text = text.replace('Р Е Ш Е Н И Е', '')
    text = text.replace('Р Е Ш И Л', '')
    words = text.split()

    # Лемматизация и удаление стоп-слов
    lemmatized_and_no_stop_words = [morph.parse(word)[0].normal_form for word in words if morph.parse(word)[0].normal_form not in stop_words_set]

    nouns = []
    loc = []
    per = []
    org = []

    # Создаем документ для NER из всего текста
    doc = Doc(text)
    doc.segment(segmenter)
    #print("Токенизация текста для NER завершена.")
    
    # Добавляем NER-теги
    doc.tag_ner(ner_tagger)
    #print("NER теги добавлены.")

    # Проверяем, какие сущности найдены
    for span in doc.spans:
        if span.type == "LOC":
            loc.append(span.text)  # Добавляем все сущности LOC
        elif span.type == "PER":
            per.append(span.text)  # Добавляем все сущности PER
        elif span.type == "ORG":
            org.append(span.text)  # Добавляем все сущности ORG


    # Добавляем все слова, которые не относятся к INFN, LOC, ORG, PER в nouns_list
    for word in lemmatized_and_no_stop_words:
        # Проверяем тип части речи с помощью pymorphy2
        parsed_word = morph.parse(word)[0]
        if parsed_word.tag.POS not in ['INFN']:
            # Не добавляем части речи: предлог, союз, частица, междометие
            if parsed_word.tag.POS != 'NPRO' and not any(word in entity_list for entity_list in [loc, per, org]):
                # Исключаем местоимения и сущности LOC, PER, ORG
                nouns.append(word) 

    nouns = [str for str in nouns if len(str) > 1]

    nouns = remove_strings_with_english(nouns)
    nouns = remove_non_russian_alpha_lines(nouns)
    per = remove_lines_with_multiple_spaces(per)

    a = []
    a.extend(loc)
    a.extend(per)
    a.extend(org)
    a.extend(nouns)

    clean_text = " ".join(a)

    end_time = time.time()
    
    execution_time = end_time - start_time
    logging.info("Извлечение данных завершено")
    logging.info(f"Время выполнения операции: {execution_time:.4f} секунд")
    return clean_text, per[-1] if bool(per) else ""


Следующий модуль, extract_data, извлекает данные из судебного дела.

Данный excel файл показывает, какие данные были извлечены из документа с помощью модуля extract_data.

In [ ]:
import pandas as pd

df = pd.read_excel('features.xlsx')
print(df)

: 

In [5]:
from typing import Tuple, Optional

import fitz
from natasha import (
        Segmenter,
    MorphVocab,
    
    NewsEmbedding,
    NewsMorphTagger,
    NewsSyntaxParser,
    NewsNERTagger,
    
    PER,
    NamesExtractor,

    Doc
)
from nltk import tokenize
import re
from collections import defaultdict
import pymorphy3


segmenter = Segmenter()
morph_vocab = MorphVocab()

emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)
syntax_parser = NewsSyntaxParser(emb)
ner_tagger = NewsNERTagger(emb)

names_extractor = NamesExtractor(morph_vocab)
morph = pymorphy3.MorphAnalyzer()

# Словарь сокращений кодексов
abbreviations = {
    "земельный": "зк", "гражданский": "гк", "трудовой": "тк", "налоговый": "нк", "кодекс об административный правонарушения": "коап", 
    "уголовный": "ук", "гражданский процессуальный": "гпк", "уголовно-процессуальный": "упк", "арбитражный процессуальный": "апк",
    "жилищный": "жк", "семейный": "ск", "бюджетный": "бк", "градостроительный": "гдк", "таможенный": "тк", 
    "кодекс административный судопроизводство": "кас", "уголовно-исполнительный": "уик", "лесной": "лк", 
    "водный": "вк", "воздушный": "вшк", "кодекс торговый мореплавание": "ктм", "кодекс внутренний водный транспорт": "кввт" 
}

def lemmatize_text(text):
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    for token in doc.tokens:
        token.lemmatize(morph_vocab)
    return " ".join(token.lemma for token in doc.tokens)

def extract_articles(text, article_pattern):
    article_dict = defaultdict(set)
    text = lemmatize_text(text)
    
    matches = article_pattern.findall(text)
    for numbers, code_type in matches:
        code_type = code_type.strip()
        article_numbers = set()
        for part in numbers.split(','):
            part = part.strip()
            if '-' in part:
                try:
                    start, end = map(int, part.split('-'))
                    article_numbers.update(range(start, end + 1))
                except ValueError:
                    continue
            else:
                found_numbers = re.findall(r'\d+', part)
                for num in found_numbers:
                    try:
                        article_numbers.add(int(num))
                    except ValueError:
                        continue

        if code_type in abbreviations:
            code_type = abbreviations[code_type]
        article_dict[code_type].update(article_numbers)
    
    return {key: sorted(value) for key, value in article_dict.items()}


def ustanovil(ust: str, text: str, resh: str) -> Tuple[Optional[str], Optional[str], Optional[str]]:
    # Разделяем текст по ust
    parts = text.split(ust, 1)
    if len(parts) != 2:
        return None, None, None

    before_ustanovil = parts[0].strip()
    after_ustanovil = parts[1]

    # Далее разделяем оставшийся текст по resh
    parts2 = after_ustanovil.split(resh, 1)
    if len(parts2) != 2:
        return None, None, None

    before_reshil = parts2[0].strip()
    reshil = before_reshil[-1000:]

    # Анализируем before_ustanovil
    doc = Doc(before_ustanovil[-700:])
    doc.segment(segmenter)
    doc.tag_ner(ner_tagger)

    orgs = [span for span in doc.spans if span.type == "ORG"]
    orgs = [span for span in orgs if span.text not in ("Арбитражный суд", "АРБИТРАЖНЫЙ СУД")]

    persons = [span for span in doc.spans if span.type == "PER"]

    first_org = min(orgs, key=lambda x: x.start) if orgs else None
    first_person = min(persons, key=lambda x: x.start) if persons else None

    if first_org and first_person:
        if first_org.start < first_person.start:
            return first_org.text, first_person.text, reshil
        else:
            return first_person.text, first_org.text, reshil
    elif len(orgs) >= 2:
        return orgs[0].text, orgs[1].text, reshil
    elif len(persons) >= 2:
        return persons[0].text, persons[1].text, reshil
    elif len(orgs) == 1 and len(persons) == 1:
        return orgs[0].text, persons[0].text, reshil
    elif len(orgs) == 1:
        return orgs[0].text, "Не найдено", reshil
    elif len(persons) == 1:
        return persons[0].text, "Не найдено", reshil
    else:
        return "Не найдено", "Не найдено", reshil


# Лишние слова для последующего удаления
dl = ["край", "края", "область", "области", "район", "района", "России", "РФ", "Российской Федерации"]

# Формы слов для последующего поиска с помощью регулярного выражения
article_declensions = ["статья", "статьи", "статей", "статье", 
                        "статьям", "статьями", "статью", "статьёй", 
                        "статьей", "статьях"]

code_declensions = ["кодекс", "кодексы", "кодекса", "кодексов",
                    "кодексу", "кодексам", "кодексом", "кодексами",
                    "кодексе", "кодексах",]

claimant_declensions = ["истец", "истцы", "истца", "истцов",
                        "истцу", "истцам", "истцом", "истцами",
                        "истце", "истцах",]

defendant_declensions = ["ответчик", "ответчики", "ответчика", "ответчиков",
                         "ответчику", "ответчикам", "ответчиком", "ответчиками",
                         "ответчике", "ответчиках"]

months = {
    "января": "01", "февраля": "02", "марта": "03", "апреля": "04", 
    "мая": "05", "июня": "06", "июля": "07", "августа": "08", 
    "сентября": "09", "октября": "10", "ноября": "11", "декабря": "12"
}

def get_data_from_file(directory: str, active_file_number: str):

    list_with_data = []

    index = active_file_number
    #print("Индекс файла: ", index)

    list_with_data.append(index)

    filename = "case_" + str(active_file_number) + ".pdf"
    doc = fitz.open(directory+filename)
    text = "\n".join([page.get_text() for page in doc])
    text = text.replace('Дело', '')
    clean_text = ' '.join(text.split())

    #print("Текст дела: ", clean_text)
    list_with_data.append(clean_text)


    clean_ver_text, judge = CleanText(directory, filename)

    #print("ОЧИЩЕННЫЙ ТЕКСТ: ")
    #print(clean_ver_text)

    #print("Судья: ", judge)
    list_with_data.append(clean_ver_text)
    list_with_data.append(judge)

    t2 = tokenize.sent_tokenize(clean_text)

    ###########################################################
    # Поиск номера дела по регулярному выражению
    case_number = re.search(r'А\d{2,}-\d+ /\d+', clean_text)
    if case_number == None:
        case_number = re.search(r'А\d{2,}-\d+/\d+', clean_text)
    #print("Номер судебного дела: ", case_number.group())

    list_with_data.append(case_number.group())

    pattern1 = r'\d{2}\s+(декабря|января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября)\s+\d{4}\s+года'

    # Паттерн для поиска даты в формате "дд.мм.гггг"
    pattern2 = r'(\d{2})\.(\d{2})\.(\d{4})'

    # Паттерн для поиска даты в формате «день месяц год г.» (в кавычках)
    pattern3 = r'«(\d{2})»\s+(декабря|января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября)\s+(\d{4})г\.'

    result1 = re.search(pattern1, text)
    result2 = re.search(pattern2, text)
    result3 = re.search(pattern3, text)

    formatted_date = None

    if result1:
        date_string = result1.group().replace("года", "").strip()
        day, month, year = date_string.split()

        month_num = months[month]
        formatted_date = f"{day}/{month_num}/{year}"

    elif result2:
        formatted_date = re.sub(pattern2, r'\1/\2/\3', result2.group())

    elif result3:
        day = result3.group(1)
        month = result3.group(2)
        year = result3.group(3)
        month_num = months[month]
        formatted_date = f"{day}/{month_num}/{year}"

    if formatted_date and len(formatted_date) > 50:
        formatted_date = 0
    
    #print("Дата: ", formatted_date)
    list_with_data.append(formatted_date)


    # Регулярное выражение для поиска статей
    article_pattern = re.compile(r"\bстатья\s+((?:\d+[,-]?\s*)+)((?:[А-Яа-я]+(?:\s+[А-Яа-я]+)*)?) кодекс российский федерация", re.IGNORECASE)
    articles = extract_articles(clean_text, article_pattern)

    list_with_data.append(articles)

    # Разбиение текста на слова
    text_splitted = text.split()

    number_or_words_in_text = len(text_splitted)

    list_with_data.append(number_or_words_in_text)

    text_splitted_clean = []
    for i in text_splitted:
        if '-' in i:
            i = i.replace('-', '')
        text_splitted_clean.append(i)
    text = " ".join(text_splitted)

    pattern = r"обоснованность заявления\s+(\w+)"

    text = text.replace('установил:', 'stop_point')
    text = text.replace('у с т а н о в и л:', 'stop_point')
    text = text.replace('УСТАНОВИЛ:', 'stop_point')
    text = text.replace('У С Т А Н О В И Л:', 'stop_point')
    text = text.replace('У с т а н о в и л:', 'stop_point')
    text = text.replace('Установил:', 'stop_point')

    text = text.replace('Р Е Ш И Л', 'start_point')
    text = text.replace('р е ш и л', 'start_point')
    text = text.replace('РЕШИЛ', 'start_point')
    text = text.replace('решил', 'start_point')
    text = text.replace('Решил', 'start_point')
    text = text.replace('Р е ш и л', 'start_point')


    t3 = tokenize.word_tokenize(text)

    claimant = ''
    defendant = ''
    reshil = ''

    claimant_found = False

    for t in t3:
            if t == "stop_point":
                claimant, defendant, reshil = ustanovil(t, text, 'start_point')
                if defendant == 'Не найдено' and claimant != 'Не найдено':
                    defendant = 'Суд'
                    list_with_data.append(claimant)
                    list_with_data.append(defendant)
                    reshil = reshil.replace('start_point', "")
                    reshil = reshil.replace('stop_point', "")
                    list_with_data.append(reshil)
                    claimant_found = True
                    break
                elif defendant == 'Не найдено' and claimant == 'Не найдено':
                    list_with_data.append(0)
                    list_with_data.append(0)
                    reshil = reshil.replace('start_point', "")
                    reshil = reshil.replace('stop_point', "")
                    list_with_data.append(reshil)
                    claimant_found = True
                    break
                else:
                    list_with_data.append(claimant)
                    list_with_data.append(defendant)
                    reshil = reshil.replace('start_point', "")
                    reshil = reshil.replace('stop_point', "")
                    list_with_data.append(reshil)
                    claimant_found = True
                    break

    if not claimant_found:
        list_with_data.append(0)
        list_with_data.append(0)
        list_with_data.append(reshil)
    loc = ''
    for i in t3:
        if i == "г.":
            y = t3.index('г.')
            loc = t3[y +1]
            list_with_data.append(loc)
            break
        elif "г." in i:
            i = i.replace('г.', '')
            loc = i
            list_with_data.append(loc)
            break

    if loc == '':
        list_with_data.append(0)

    decision = ''
    doc = ''
    match = re.search(pattern, text)
    if match:
        filtered_loc = match.group(1)
        #decision = "в пользу истца"
        decision = "принято"
        list_with_data.append(decision)
    else:
        if 'start_point' in text:
            if "частично" in text:
                decision = "частично"
                list_with_data.append(decision)
            elif "отказать" in text or "оставить без" in text:
                decision = "отказано"
                list_with_data.append(decision)
            elif "удовлетворении" in text or "признать обоснованным" in text or "удовлетворить" in text:
                decision = "принято"
                list_with_data.append(decision)
            elif "в пользу":
                parts = text.split("в пользу", 1)
                if len(parts) > 1:
                    t2 = parts[1].strip()
                    doc = Doc(t2)
                    doc.segment(segmenter)
                    doc.tag_ner(ner_tagger)
                    orgs = [span.text for span in doc.spans if span.type == "ORG"]
                    persons = [span.text for span in doc.spans if span.type == "PER"]
                    #print("ФАЙЛ ", active_file_number)
                    if orgs or persons:
                        if orgs!= [] and defendant == orgs[0]:
                            #decision = "в пользу ответчика"
                            decision = "отказано"
                            list_with_data.append(decision)
                        elif orgs!= [] and claimant == orgs[0]:
                            #decision = "в пользу истца"
                            decision = "принято"
                            list_with_data.append(decision)
                        elif persons != [] and claimant == persons[0]:
                            #decision = "в пользу истца"
                            decision = "принято"
                            list_with_data.append(decision)
                        else:
                            #decision = "в пользу ответчика"
                            decision = "отказано"
                            list_with_data.append(decision)
            else:
                list_with_data.append(0)
    return list_with_data

dict = {}

def get_dict_with_data(directory: str, number_of_docs: int):
    for i in range (1, number_of_docs + 1):

        active_file_number = str(i)
        list_with_data = get_data_from_file(directory, active_file_number)
        #print("ФАЙЛ " + active_file_number + " ОБРАБОТАН")
        dict[active_file_number] = list_with_data
    return dict

Модуль export_to_excel экспортирует нужные нам данные в excel файл. Данные будут использоваться для обучения алгоритма и тестируемой выборки.

In [ ]:
import os
import pandas as pd

def create_empty_excel(columns: list, filename: str, sheet_name: str = 'Sheet1'):
    df = pd.DataFrame(columns=columns)

    if not os.path.exists('excel_files'):
        os.makedirs('excel_files')

    filepath = os.path.join('excel_files', filename)
    excel_writer = pd.ExcelWriter(filepath, engine='xlsxwriter')
    df.to_excel(excel_writer, index=False, sheet_name=sheet_name, freeze_panes=(1, 0))
    excel_writer._save()

    return filepath, df

columns = ['index', 'text', 'clean_text', 'judge', 'case_number', 'publication_date', 'articles', 'number_of_words_in_text', 'claimant', 'defendant', 'research_data', 'location', 'decision']

def create_table(filename: str):
    filepath, df = create_empty_excel(columns=columns,
                                  filename=filename)
    return df, filepath

def make_excel_file(number_of_files: int, excel_directory: str, df, dict):
    for i in range(1, number_of_files + 1):
        current_row = i
        df.loc[current_row] = [None] * len(columns)
        active_file_number = str(i)
        x = dict[active_file_number]
        u = 0
        for u, y in enumerate(x):
            if u < len(columns):
                df.at[current_row, columns[u]] = y
    df.to_excel(excel_directory, index=False)

directory = "./pdf_cases/"
filename='data_for_training.xlsx'


df, filepath = create_table(filename)
number_of_files = 100

dict = get_dict_with_data(directory, number_of_files)


active_file_number = ''
excel_directory = 'excel_files/data_for_training.xlsx'


make_excel_file(number_of_files, excel_directory, df, dict)

# Показать первые 5 строк 
print("Данные для обучения")
df.head()

Данные для обучения


,index,text,clean_text,judge,case_number,publication_date,articles,number_of_words_in_text,claimant,defendant,research_data,location,decision
1,1,АРБИТРАЖНЫЙ СУД ВЛАДИМИРСКОЙ ОБЛАСТИ Именем Ро...,ВЛАДИМИРСКОЙ ОБЛАСТИ Российской Федерации Влад...,В.В.Романова,А11-9330/2021,28/01/2022,"{'апк': [49, 110, 167, 168, 169, 170, 171, 176...",1899,0,0,,Владимир,отказано
2,2,АРБИТРАЖНЫЙ СУД СТАВРОПОЛЬСКОГО КРАЯ Именем Ро...,СТАВРОПОЛЬСКОГО КРАЯ Ставропольского края Став...,И.В. Навакова,А63-14100/2024,18/11/2024,"{'апк': [167, 168, 169, 170, 198, 201]}",5487,Наваковой И.В.,Федеральной налоговой службы,"кция представила доказательства, что налогопла...",Буденновск,частично
3,3,117_42015355 АРБИТРАЖНЫЙ СУД ГОРОДА МОСКВЫ 115...,ГОРОДА МОСКВЫ Москва Большая Тульская РОССИЙСК...,Елена Алибековна,А40-96629/24,28/12/2024,{},808,Председательствующего,ИНДИВИДУАЛЬНОГО ПРЕДПРИНИМАТЕЛЯ МИРОШНИЧЕНКО И...,"тельствует о том, что Маркетплейс возвращены о...",Москва,частично
4,4,36_16863791 Арбитражный суд Московской области...,Московской области Москва Российской Федерации...,А.О. Монгуш,А41-99065/24,28/12/2024,"{'апк': [121, 167, 168, 169, 170, 176, 223]}",1736,РЕШЕНИЕ,Монгуш А.О.,нению в отношении должника положения пункта 8 ...,Москва,принято
5,5,112_42038364 АРБИТРАЖНЫЙ СУД ГОРОДА МОСКВЫ 115...,ГОРОДА МОСКВЫ Москва Большая Тульская РОССИЙСК...,Т.В. Моисеенко,А40-187641/24,28/12/2024,{'гк': [10]},1720,Моисеенко Т.В,"ООО ""РУСЬ""",ания для грузополучателей. При этом необходимо...,Москва,частично


In [14]:
directory = "./pdf_cases_for_testing/"
filename='data_for_testing.xlsx'

df, filepath = create_table(filename)

number_of_files = 50

dict = get_dict_with_data(directory, number_of_files)

active_file_number = ''
excel_directory = 'excel_files/data_for_testing.xlsx'

make_excel_file(number_of_files, excel_directory, df, dict)

print("Тестовая выборка")
df.head()

Тестовая выборка


,index,text,clean_text,judge,case_number,publication_date,articles,number_of_words_in_text,claimant,defendant,research_data,location,decision
1,1,АРБИТРАЖНЫЙ СУД ВОРОНЕЖСКОЙ ОБЛАСТИ Р Е Ш Е Н ...,ВОРОНЕЖСКОЙ ОБЛАСТИ Воронеж ...,В.А. Козлов,А14-12621/2017,20/12/2017,{},2021,Греченко Дмитрию Александровичу,Управления Пенсионного фонда,ица (индивидуального предпринимателя) и сведен...,Воронеж,отказано
2,2,5_6675402 Арбитражный суд Московской области 1...,Московской области Москва Москва Московской об...,П.М. Морхат,А41-91338/17,19/01/2018,"{'апк': [8, 9, 65, 69, 71, 121, 223, 319]}",1423,Морхата П.М.,ООО «ЦЕНТР ФИНАНСОВЫХ РАССЛЕДОВАНИЙ» (ИНН 7714...,"ика было принято решение о ликвидации, следова...",Москва,принято
3,3,Арбитражный суд Краснодарского края Именем Рос...,Краснодарского края Российской Федерации Красн...,В.А. Язвенко,А32-49408/2017,12/07/2010,"{'апк': [4, 121], 'гк': [12]}",1934,Язвенко В.А.,Администрации муниципального образования,о его завершении. Поскольку производство по де...,Краснодар,None
4,4,2111311_3097528 1 АРБИТРАЖНЫЙ СУД РЕСПУБЛИКИ Б...,РЕСПУБЛИКИ БАШКОРТОСТАН Республика Башкортоста...,И.Р. Юсеева,А07-29386/17,09/01/2018,"{'апк': [9, 65, 70, 110, 156, 167, 168, 169, 1...",1538,0,0,ты /п. 23 Постановления Пленума Верховного Суд...,Уфа,принято
5,5,41/2017-154055(1) АРБИТРАЖНЫЙ СУД ОМСКОЙ ОБЛАС...,ОМСКОЙ ОБЛАСТИ Омск Российской Федерации Омск ...,Воронов Тимур Александрович,А46-9190/2017,31/12/2017,"{'апк': [65, 110, 156, 167, 168, 169, 170, 171...",1814,0,0,,Омск,отказано


Следующим шагом было конвертирование excel файлов с данными в csv формат.

In [27]:
data_xls = pd.read_excel('./excel_files/data_for_training.xlsx', 'Sheet1', index_col=None)
data_xls.to_csv('./csv/arbitr_dataset_for_training.csv', encoding='utf-8')

data_xls = pd.read_excel('./excel_files/data_for_testing.xlsx', 'Sheet1', index_col=None)
data_xls.to_csv('./csv/arbitr_dataset_for_testing.csv', encoding='utf-8')

def convert(filename:str):
    filename_new = './excel_files/' + filename + '.xlsx'
    data_xls = pd.read_excel(filename_new, 'Sheet1', index_col=None)
    csv_name = './csv/' + filename + '.csv'
    data_xls.to_csv(csv_name, encoding='utf-8')

Модуль prepare_csv подготавливает наши данные.

In [ ]:
import csv


def read_cell(x, y, filename):
    with open(filename, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        y_count = 0
        for n in reader:
            if y_count == y:
                cell = n[x]
                return cell
            y_count += 1

def prepare_csv(filename: str, number_of_files: int):
    full_filename = './csv/' + filename + '.csv'
    filename_new = './csv/prepared_' + filename + '.csv'
    with open(filename_new, 'w', newline='', encoding='utf-8') as file:

        writer = csv.writer(file)
        field = ["data", "decision", "judge", "region", "articles"]

        writer.writerow(field)
        for i in range(1, number_of_files + 1):
            text = read_cell(11, i, full_filename)
            dec = read_cell(13, i, full_filename)
            judge = read_cell(4, i, full_filename)
            region = read_cell(12, i, full_filename)
            articles = read_cell(7, i, full_filename)
            if text == '' or dec == '' or text == None or dec == None:
                pass
            else:

                writer.writerow([text, dec, judge, region, articles])

        print('\ncsv Данные были очищены и скопированы в таргетированную папку')

Очищаем извлеченные данные с помощью функции prepare_csv.

In [29]:
filename_train_data = 'arbitr_dataset_for_training'
prepare_csv(filename_train_data, 250)
df = pd.read_csv('./csv/arbitr_dataset_for_training.csv')
print("Подготовленный обучаемый набор данных")
df.head()


csv Данные были очищены и скопированы в таргетированную
Подготовленный обучаемый набор данных


,Unnamed: 0,index,text,clean_text,judge,case_number,publication_date,articles,number_of_words_in_text,claimant,defendant,research_data,location,decision
0,0,1,АРБИТРАЖНЫЙ СУД ВЛАДИМИРСКОЙ ОБЛАСТИ Именем Ро...,ВЛАДИМИРСКОЙ ОБЛАСТИ Российской Федерации Влад...,В.В.Романова,А11-9330/2021,28/01/2022,"{'апк': [49, 110, 167, 168, 169, 170, 171, 176...",1899,0,0,NaN,Владимир,отказано
1,1,2,АРБИТРАЖНЫЙ СУД СТАВРОПОЛЬСКОГО КРАЯ Именем Ро...,СТАВРОПОЛЬСКОГО КРАЯ Ставропольского края Став...,И.В. Навакова,А63-14100/2024,18/11/2024,"{'апк': [167, 168, 169, 170, 198, 201]}",5487,Наваковой И.В.,Федеральной налоговой службы,"кция представила доказательства, что налогопла...",Буденновск,частично
2,2,3,117_42015355 АРБИТРАЖНЫЙ СУД ГОРОДА МОСКВЫ 115...,ГОРОДА МОСКВЫ Москва Большая Тульская РОССИЙСК...,Елена Алибековна,А40-96629/24,28/12/2024,{},808,Председательствующего,ИНДИВИДУАЛЬНОГО ПРЕДПРИНИМАТЕЛЯ МИРОШНИЧЕНКО И...,"тельствует о том, что Маркетплейс возвращены о...",Москва,частично
3,3,4,36_16863791 Арбитражный суд Московской области...,Московской области Москва Российской Федерации...,А.О. Монгуш,А41-99065/24,28/12/2024,"{'апк': [121, 167, 168, 169, 170, 176, 223]}",1736,РЕШЕНИЕ,Монгуш А.О.,нению в отношении должника положения пункта 8 ...,Москва,принято
4,4,5,112_42038364 АРБИТРАЖНЫЙ СУД ГОРОДА МОСКВЫ 115...,ГОРОДА МОСКВЫ Москва Большая Тульская РОССИЙСК...,Т.В. Моисеенко,А40-187641/24,28/12/2024,{'гк': [10]},1720,Моисеенко Т.В,"ООО ""РУСЬ""",ания для грузополучателей. При этом необходимо...,Москва,частично


In [30]:
filename_test_data = 'arbitr_dataset_for_testing'
prepare_csv(filename_test_data, 50)
df = pd.read_csv('./csv/arbitr_dataset_for_testing.csv')
print("Подготовленная тестовая выборка")
df.head()


csv Данные были очищены и скопированы в таргетированную
Подготовленная тестовая выборка


,Unnamed: 0,index,text,clean_text,judge,case_number,publication_date,articles,number_of_words_in_text,claimant,defendant,research_data,location,decision
0,0,1,АРБИТРАЖНЫЙ СУД ВОРОНЕЖСКОЙ ОБЛАСТИ Р Е Ш Е Н ...,ВОРОНЕЖСКОЙ ОБЛАСТИ Воронеж ...,В.А. Козлов,А14-12621/2017,20/12/2017,{},2021,Греченко Дмитрию Александровичу,Управления Пенсионного фонда,ица (индивидуального предпринимателя) и сведен...,Воронеж,отказано
1,1,2,5_6675402 Арбитражный суд Московской области 1...,Московской области Москва Москва Московской об...,П.М. Морхат,А41-91338/17,19/01/2018,"{'апк': [8, 9, 65, 69, 71, 121, 223, 319]}",1423,Морхата П.М.,ООО «ЦЕНТР ФИНАНСОВЫХ РАССЛЕДОВАНИЙ» (ИНН 7714...,"ика было принято решение о ликвидации, следова...",Москва,принято
2,2,3,Арбитражный суд Краснодарского края Именем Рос...,Краснодарского края Российской Федерации Красн...,В.А. Язвенко,А32-49408/2017,12/07/2010,"{'апк': [4, 121], 'гк': [12]}",1934,Язвенко В.А.,Администрации муниципального образования,о его завершении. Поскольку производство по де...,Краснодар,NaN
3,3,4,2111311_3097528 1 АРБИТРАЖНЫЙ СУД РЕСПУБЛИКИ Б...,РЕСПУБЛИКИ БАШКОРТОСТАН Республика Башкортоста...,И.Р. Юсеева,А07-29386/17,09/01/2018,"{'апк': [9, 65, 70, 110, 156, 167, 168, 169, 1...",1538,0,0,ты /п. 23 Постановления Пленума Верховного Суд...,Уфа,принято
4,4,5,41/2017-154055(1) АРБИТРАЖНЫЙ СУД ОМСКОЙ ОБЛАС...,ОМСКОЙ ОБЛАСТИ Омск Российской Федерации Омск ...,Воронов Тимур Александрович,А46-9190/2017,31/12/2017,"{'апк': [65, 110, 156, 167, 168, 169, 170, 171...",1814,0,0,NaN,Омск,отказано


Модуль rf_kmeans_pipeline связывает модели машинного обучения в единый пайплайн и выводит нам метрики с возможным исходом судебного дела.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix
from imblearn.over_sampling import RandomOverSampler


# Загрузка данных
df = pd.read_csv('./csv/prepared_arbitr_dataset_for_training.csv')

texts = df['data']  # Тексты для обучения
labels = df['decision'].tolist()  # Метки решений

# Кастомный трансформер для добавления кластеров
class ClusterTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=20, random_state=42):
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.kmeans = KMeans(n_clusters=n_clusters, n_init='auto', random_state=random_state)
    
    def fit(self, X, y=None):
        self.kmeans.fit(X)
        return self
    
    def transform(self, X):
        clusters = self.kmeans.predict(X)
        # Преобразование X (матрица признаков) в DataFrame и добавляем кластерные метки
        X_with_clusters = pd.DataFrame(X.toarray())
        X_with_clusters['cluster'] = clusters
        X_with_clusters.columns = X_with_clusters.columns.astype(str)
        return X_with_clusters

# Создание пайплайна
pipeline_with_random_oversampler = ImbPipeline([
    ('tfidf', TfidfVectorizer()),
    ('cluster', ClusterTransformer()),
    ('oversample', RandomOverSampler(random_state=42)),
    ('clf', RandomForestClassifier(n_estimators=100, bootstrap=True, class_weight='balanced', 
                                   max_features=6, max_depth=100, random_state=42))
])

pipeline_with_random_oversampler.fit(texts, labels)
df2 = pd.read_csv('./csv/prepared_arbitr_dataset_for_testing.csv')
new_texts = df2['data']
new_labels = df2['decision']

# Предсказания
predictions = pipeline_with_random_oversampler.predict(new_texts)

# Метрики
accuracy = accuracy_score(new_labels, predictions)
precision = precision_score(new_labels, predictions, average='weighted')
recall = recall_score(new_labels, predictions, average='weighted')
f1 = f1_score(new_labels, predictions, average='weighted')

# Печать результатов
print(f"Accuracy: {accuracy:.2f}")
print("Precision Score: ", precision)
print("Recall Score: ", recall)
print("F1 Score: ", f1)

def show_new_prediction(filename: str):
    new_filename = './csv/prepared_' + filename + '.csv'
    df3 = pd.read_csv(new_filename)
    new_texts = df3['data']
    new_labels = df3['decision']

    predictions = pipeline_with_random_oversampler.predict(new_texts)
    print(predictions[0])
    
    return predictions[0]

Accuracy: 0.72
Precision Score:  0.7517361111111112
Recall Score:  0.71875
F1 Score:  0.7131493506493507


Примечание: данная работа представляет работу алгоритма в виде статьи. Пользовательский интерфейс, визуализация частотности слов и итоговый отчет не были показаны.